# hydrogen 2d

In [ ]:
import math
import cupy as np

from qiskit_aer import AerSimulator

from crsq.blocks.antisymmetrization import AntisymmetrizationSpec
from crsq.blocks.discretization import DiscretizationSpec
from crsq.blocks.energy_initialization import EnergyConfigurationSpec
from crsq.blocks.hamiltonian import HamiltonianSpec
from crsq.blocks.time_evolution.spec import TimeEvolutionSpec
from crsq.blocks.wave_function import WaveFunctionRegisterSpec
from crsq.blocks.time_evolution.suzuki_trotter import SuzukiTrotterMethodBlock

from crsq.blocks import ( hamiltonian, antisymmetrization, time_evolution, wave_function, energy_initialization, discretization, rfqhamiltonian)

from crsq.models.hydrogen2d import PsiH2D

dim = 2  # 1 dimension
n1 = 5 # bits per coordinate
M=1<<n1
L = 16 # 16 bohr
eta = 1 # num of electrons
Ln = 0 # moving nucleus
Ls = 1 # stationary nucleus
antisym_method = 3 # binary coded antisymmetrization method
wfr_spec = WaveFunctionRegisterSpec(dim, n1, L, eta, Ln, Ls)

delta_t = 0.001 # a.u.
disc_spec = DiscretizationSpec(delta_t)
asy_spec = AntisymmetrizationSpec(wfr_spec, antisym_method)
nuclei_data = [{"mass": 1680, "charge": 1, "pos": (0, 0)}]

ham_spec = HamiltonianSpec(wfr_spec, nuclei_data=nuclei_data)

M = 2**n1
dq = L/M

# quantum numbers
qn_n = 0
qn_m = 0
psifunc2 = PsiH2D(0, 0, dq/2, qn_n, qn_m)

xv = np.zeros((M, M))
yv = np.zeros((M, M))
for i in range(M):
    yv[:,i] = np.linspace(-M//2, M//2-1, M)
    xv[i,:] = np.linspace(-M//2, M//2-1, M)
# discretized wave number values
# value of V
qxv = xv * dq
qyv = yv * dq

psi0 = dq*psifunc2(qxv, qyv)

# ini_dims = [psix]
ini_electrons = [psi0]
ini_configs = [ini_electrons]
initial_electron_orbitals = ini_configs

initial_nucleus_orbitals = [[]]
ene_spec = EnergyConfigurationSpec([1], initial_electron_orbitals, initial_nucleus_orbitals)

def potential_func(r):
    return -1/(r+0.2)

rfq_spec = rfqhamiltonian.RfqPotentialSpec(wfr_spec, potential_func, potential_func)

num_nucl_it = 1 # number of nucleus iterations
# num_elec_it = 10 # number of electron iterations
num_elec_it = 1 # number of electron iterations
evo_method = time_evolution.SUZUKI_TROTTER_QROM
evo_spec = TimeEvolutionSpec(ham_spec, disc_spec, num_nucl_it, num_elec_it, method=evo_method, rfq_spec=rfq_spec)

stm = SuzukiTrotterMethodBlock(evo_spec, ene_spec, asy_spec, use_motion_block_gates=True)

stm.circuit.draw(output='mpl', scale=0.6)


In [ ]:
emb = stm.build_electron_motion_block()
epqb = emb.build_elec_potential_block_qrom()
epqb.circuit.draw(output='mpl', scale=0.6)

# execute

In [ ]:
from qiskit import transpile

backend = AerSimulator()
code = transpile(stm.circuit, backend)
results = backend.run(code).result()